# Single Bias-Experiment Runner (Multi Class)

Runs one bias-evaluation experiment end-to-end. Accepts a YAML config file specifying the bias type (length / position / sycophancy / uncertainty), reward model, dataset source, and output directories, as argument to its `run(...)` function.

Should run on any `transformers`-supported GPU (Nvidia, Apple Silicon) or CPU (we discourage use of CPUs due to very high inference latency) if you use the multi-GPU branch.

This particular example required ~4 GB on a T4 GPU and took less than 30 minutes to run end-to-end.

**Summary:**

- Loads the corresponding `BiasExperiment` subclass, trains a linear probe on a held-out probe set, evaluates on the test split, saves the probe artifact, and writes diagnostic plots.
- Returns an `ExperimentResults` object containing summary statistics of the run.
- Handles cross-dataset generalisation by pointing the probe trainer at one dataset and the evaluator at another.

In [ ]:
!nvidia-smi

In [ ]:
# Clone/pull the `bugfix/multi-class` branch
!git clone -b bugfix/multi-class https://github.com/nondatur/OneBiasAfterAnotherFork.git
!cd OneBiasAfterAnotherFork && git status && git pull && git log -1

Set these variables to specify the location of HF artifacts and the codebase:

In [ ]:
# We specify these two global variables:
HF_HOME   = "<full path to your huggingface cache directory>"
REPO_HOME = "<full path to your repository directory>"

In [ ]:
# login to HF with custom creds
from huggingface_hub import login
try:
    HF_TOKEN = "<your huggingface token>" # Frankfurt AI Safety token
    login(token=HF_TOKEN)
    print("Successfully logged in to Hugging Face.")
except Exception as e:
    print(f"Login failed: {e}")
    print("Please ensure you have added 'HF_TOKEN' to your Colab secrets.")

In [ ]:
import sys
import os
from pathlib import Path

# Set up HF home (for Colab only)
os.environ['HF_HOME'] = HF_HOME
if HF_HOME not in sys.path:
    sys.path.append(HF_HOME)

# Ensure the repository root is in the system path
repo_path = os.path.abspath(REPO_HOME)
if repo_path not in sys.path:
     sys.path.append(repo_path)

# Set up the project root
PROJECT_ROOT = Path(REPO_HOME).resolve()
if str(PROJECT_ROOT) not in sys.path:
     sys.path.insert(0, str(PROJECT_ROOT))

# Check system path(s)
for path in sys.path:
    print(f"Current sys.path(s): {path}")

In [ ]:
try:
  from src.nb.experiments.position import run_position_experiment # pyright: ignore[reportMissingImports]
  from src.nb.experiments.base import ExperimentConfig # pyright: ignore[reportMissingImports]
  from src.nb.experiments.position import PositionBiasExperiment # pyright: ignore[reportMissingImports]
  from src.nb.datasets.position import PositionMultiClassDataset # pyright: ignore[reportMissingImports]
  from src.nb.datasets.muti_class_parsing import ( # pyright: ignore[reportMissingImports]
      format_multi_class_prompt,
      format_multi_class_response,
      parse_to_nchoice_mcq,
      validate_class_labels,
  )
except ImportError:
  print(f"Import failed: {e}")
  print("Please ensure you have added the project root to your Python path.")

In [ ]:
def run_programmatic():
    """Instantiate dataset and experiment programmatically."""

    # Step 1: Create config
    config = ExperimentConfig(

        #name="my_custom_2class_position_experiment_deberta", # DeBERTa experiment
        name="my_custom_2class_position_experiment_llama3", # Llama 3.1 experiment

        bias_type="position",

        #model_path="OpenAssistant/reward-model-deberta-v3-large-v2", # DeBERTa checkpoint
        model_path="skywork/Skywork-Reward-Llama-3.1-8B", # Llama 3.1 checkpoint

        dataset_source="frankfurt-ai-safety/toxicity-classification-mc",
        dataset_class="position_multi_class",

        probe_size=500,
        max_test_examples=500,

        batch_size=8,

        #max_length=2048, # can be used with datasets with short texts and/or or small models
        max_length=512, # use to prevent OOM errors when loading models with billions of params

        device="auto", # set on "auto" for machines with >= 0 NVidia GPUs

        extra={
            "dataset_id": "frankfurt-ai-safety/toxicity-classification-mc",
            "train_split": "train",
            "eval_split": "test",
            "num_classes": 2,  # Use 2 classes (minimum, with a maximum of 4)
            "class_labels": ["SAFE", "UNSAFE"],
            "clean_with_correctness": True,
        },

    )

    print('\n-----------------------------')
    print('Running Experiment')
    print('-----------------------------\n')

    # Step 2: Create experiment
    experiment = PositionBiasExperiment(config)

    # Step 3: Load model and dataset
    print("\nLoading model...")
    experiment.load_model()
    print("\nLoading dataset...")
    experiment.load_dataset()

    # Step 4: Run experiment
    print("\n----------------------")
    print("Training and testing")
    print("----------------------\n")
    results = experiment.run() # run: experiment.evaluate() to skip training when probe is already trained

    print(f"\nDataset: {experiment.dataset.name}")
    print(f"Position labels: {experiment.dataset.position_labels}")
    print(f"Accuracy: {results.baseline_metrics['accuracy']:.2%}")

    return results

In [ ]:
def explore_dataset_directly():
    """Directly instantiate the multi-class dataset to explore it."""

    # Read dataset with custom labels
    dataset = PositionMultiClassDataset(
        source="frankfurt-ai-safety/toxicity-classification-mc",
        split="train",
        eval_split="test",
        probe_size=100,
        num_classes=2,
        class_labels=["SAFE", "UNSAFE"],
    )

    print('\n-----------------------------')
    print('Exploring Dataset')
    print('-----------------------------\n')
    print(f"\nDataset type: {dataset.name}")
    print(f"Dataset name: {dataset.source}")
    print(f"Train data: {dataset.train_split}")
    print(f"Test split: {dataset.eval_split}")
    print(f"Position labels: {dataset.position_labels}")
    print(f"Number of classes: {dataset.num_classes}")
    print(f"Split seed: {dataset.split_seed}")
    print(f"Probe size: {dataset.probe_size}")
    print(f"Max test examples: {dataset.max_test_examples}")

    # Load and inspect raw data
    raw_data = dataset._load_raw_data()
    print(f"\nLoaded {len(raw_data)} examples")

    first_example = raw_data[0]
    print(f"\nFirst example:")
    print(f"  Question: {first_example['question']}")
    print(f"  Choices: {first_example['choices']}")
    print(f"  Correct idx: {first_example['correct_idx']}")

In [ ]:
explore_dataset_directly()

In [ ]:
experiment = run_programmatic()

In [ ]:
import json
print(json.dumps(experiment.to_dict(), indent=4))